# Sistem Rekomendasi Film

![](https://c.files.bbci.co.uk/D1E0/production/_106082735_mediaitem106082731.jpg)

Sistem rekomendasi film menggunakan dataset ml-latest-small yang merupakan sebuah kumpulan data yang berisi informasi mengenai rating film bintang 5 dan aktivitas tagging bebas dari layanan rekomendasi film MovieLens. Dataset ini dirancang sebagai dataset pengembangan dan dirilis pada 26 September 2018.

Data ini mencakup 100.836 rating dan 3.683 aplikasi tag yang diberikan oleh 610 pengguna terhadap 9.742 film. Aktivitas pengguna ini tercatat antara 29 Maret 1996 dan 24 September 2018.

Sumber : https://files.grouplens.org/datasets/movielens/

# Import Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Data Understanding

## Mendownload datasets

Medownload datasets yang digunakan dari https://files.grouplens.org/datasets/movielens/ml-latest-small.zip



In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

--2025-05-27 14:11:36--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.65.152
Connecting to files.grouplens.org (files.grouplens.org)|128.101.65.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K   737KB/s    in 1.3s    

2025-05-27 14:11:39 (737 KB/s) - ‘ml-latest-small.zip’ saved [978202/978202]



## Unzip file datasets yang sudah didownload

In [4]:
!unzip /content/ml-latest-small.zip

Archive:  /content/ml-latest-small.zip
   creating: ml-latest-small/
  inflating: ml-latest-small/links.csv  
  inflating: ml-latest-small/tags.csv  
  inflating: ml-latest-small/ratings.csv  
  inflating: ml-latest-small/README.txt  
  inflating: ml-latest-small/movies.csv  


Dari informasi tersebut dapat diketahui bahwa dalam file datasets terdapat 4 file csv yaitu links, tags, ratings, dan movies yang nantinya akan digunakan dalam pembuatan sistem rekomendasi.

## Membaca jumlah data yang tersedia

In [5]:
links = pd.read_csv('/content/ml-latest-small/links.csv')
movies = pd.read_csv('/content/ml-latest-small/movies.csv')
ratings = pd.read_csv('/content/ml-latest-small/ratings.csv')
tags = pd.read_csv('/content/ml-latest-small/tags.csv')

print('Jumlah data sumber film: ', len(links.movieId.unique()))
print('Jumlah data film: ', len(movies.movieId.unique()))
print('Jumlah data rating film: ', len(ratings.movieId.unique()))
print('Jumlah data tag film: ', len(tags.userId.unique()))

Jumlah data sumber film:  9742
Jumlah data film:  9742
Jumlah data rating film:  9724
Jumlah data tag film:  58


# Univariate Exploratory Data Analysis

## Variabel-variabel pada movie dataset



*   **Links** : Merupakan sumber id film
*   **Ratings** : Merupakan review pengguna
*   **Tags** : Merupakan tag/ label yang berkaitan dengan film
*   **Movies**: Merupakan informasi mengenai film





## Links variabel

In [6]:
links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


Berdasarkan output di atas, kita dapat mengetahui bahwa file `links.csv` memiliki **9742 entri**. File ini merupakan bagian dari dataset MovieLens, yang berkaitan dengan **film**. Oleh karena itu, terdapat tiga variabel dalam file ini, yaitu:

* **movieId**: ID unik untuk setiap film dalam dataset MovieLens.
* **imdbId**: ID film pada basis data IMDb.
* **tmdbId**: ID film pada basis data TMDb (The Movie Database).


## Movies variabel

In [7]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


Berdasarkan output di atas, kita dapat mengetahui bahwa file `movies.csv` memiliki **9742 entri**. File ini termasuk dalam kategori data **film**, yang merupakan bagian dari dataset MovieLens. Terdapat tiga variabel pada file ini, yaitu:

* **movieId**: ID unik untuk setiap film.
* **title**: Judul film.
* **genres**: Genre atau kategori film (dapat terdiri dari lebih dari satu genre dan dipisahkan oleh tanda "|").


## Ratings variabel

In [8]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


Berdasarkan output di atas, kita dapat mengetahui bahwa file `ratings.csv` memiliki **100.836 entri**. File ini termasuk dalam kategori data **rating film** oleh pengguna dalam dataset MovieLens. Terdapat empat variabel pada file ini, yaitu:

* **userId**: ID pengguna yang memberikan rating.
* **movieId**: ID film yang diberikan rating.
* **rating**: Nilai rating yang diberikan pengguna terhadap film (dalam skala bintang 0.5–5.0).
* **timestamp**: Waktu saat rating diberikan.


In [9]:
ratings.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


Berdasarkan output statistik deskriptif dari file `ratings.csv` di atas, berikut adalah ringkasan informasi penting untuk masing-masing kolom:

* Rating berkisar dari **0.5 hingga 5.0**, dengan rata-rata **3.5**.
* Kuartil menunjukkan bahwa sebagian besar rating berkisar antara 3.0 hingga 4.0, yang berarti pengguna cenderung memberikan penilaian positif.

Dari informasi ini, kita dapat menyimpulkan bahwa data memiliki distribusi rating yang cukup normal, dengan mayoritas pengguna memberikan rating sedang hingga tinggi.

## Tag variabel

In [10]:
print(tags.shape)

(3683, 4)


In [11]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


Berdasarkan output `tags.shape` di atas, kita dapat mengetahui bahwa file `tags.csv` memiliki **3.683 entri** dan **4 kolom**.

File ini termasuk dalam kategori data tambahan (metadata) yang memperkaya informasi mengenai film. Secara umum, data ini kemungkinan berisi variabel seperti:

* **userId** – ID pengguna yang memberikan tag.
* **movieId** – ID film yang ditandai.
* **tag** – Kata kunci atau label yang diberikan oleh pengguna.
* **timestamp** – Waktu pemberian tag.

Data `tags` ini berguna untuk menambahkan konteks atau opini pengguna terhadap film, yang bisa dimanfaatkan dalam sistem rekomendasi atau analisis sentimen.


# Data Preprocessing

## Menggabungkan beberapa file data film

In [12]:
movie_all = np.concatenate((
    links.movieId.unique(),
    movies.movieId.unique(),
    ratings.movieId.unique(),
    tags.movieId.unique(),
))

movie_all = np.sort(np.unique(movie_all))

print('Jumlah seluruh data film berdasarkan movieId: ', len(movie_all))

Jumlah seluruh data film berdasarkan movieId:  9742


Setelah digabung data keseluruhan berjumlah 9742 baris

## Menggabungkan Seluruh User

In [13]:
user_all = np.concatenate((
        ratings.userId.unique(),
        tags.userId.unique()
))

user_all = np.sort(np.unique(user_all))
print('Jumlah seluruh user: ', len(user_all))

Jumlah seluruh user:  610


Setelah digabungkan dan diurutkan ternyata terdapat 610 pengguna yang memberikan review terhadap film

## Mengetahui Jumlah Rating

In [14]:
movie_info = pd.concat([links, movies, tags])

movie = pd.merge(ratings, movie_info , on='movieId', how='left')
movie

,userId_x,movieId,rating,timestamp_x,imdbId,tmdbId,title,genres,userId_y,tag,timestamp_y
0,1,1,4.0,964982703,114709.0,862.0,NaN,NaN,NaN,NaN,NaN
1,1,1,4.0,964982703,NaN,NaN,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,NaN,NaN,NaN
2,1,1,4.0,964982703,NaN,NaN,NaN,NaN,336.0,pixar,1.139046e+09
3,1,1,4.0,964982703,NaN,NaN,NaN,NaN,474.0,pixar,1.137207e+09
4,1,1,4.0,964982703,NaN,NaN,NaN,NaN,567.0,fun,1.525286e+09
...,...,...,...,...,...,...,...,...,...,...,...
434880,610,168252,5.0,1493846352,NaN,NaN,NaN,NaN,567.0,gritty,1.525284e+09
434881,610,168252,5.0,1493846352,NaN,NaN,NaN,NaN,567.0,heartbreaking,1.525284e+09
434882,610,168252,5.0,1493846352,NaN,NaN,NaN,NaN,567.0,predictible plot,1.525284e+09
434883,610,170875,3.0,1493846415,4630562.0,337339.0,NaN,NaN,NaN,NaN,NaN


Terlihat setelah digabungkan terdapat banyak sekali data hilang atau kosong

In [15]:
movie.isnull().sum()

,0
userId_x,0
movieId,0
rating,0
timestamp_x,0
imdbId,334049
tmdbId,334062
title,334049
genres,334049
userId_y,201672
tag,201672


Kolom dengan missing value atau kehilangan data terbanyak pada kolom imdbId, tmdbid, title, genres, userId, tag & timestamp

In [16]:
movie.groupby('movieId').sum()

,userId_x,rating,timestamp_x,imdbId,tmdbId,title,genres,userId_y,tag,timestamp_y
movieId,,,,,,,,,,
1,329520,4215.0,1214572277395,24662435.0,185330.0,Toy Story (1995)Toy Story (1995)Toy Story (199...,Adventure|Animation|Children|Comedy|FantasyAdv...,296055.0,pixarpixarfunpixarpixarfunpixarpixarfunpixarpi...,8.173308e+11
2,217506,2265.0,749631499932,12484670.0,972840.0,Jumanji (1995)Jumanji (1995)Jumanji (1995)Juma...,Adventure|Children|FantasyAdventure|Children|F...,72600.0,fantasymagic board gameRobin Williamsgamefanta...,6.296298e+11
3,58988,678.0,209062937544,5887856.0,811304.0,Grumpier Old Men (1995)Grumpier Old Men (1995)...,Comedy|RomanceComedy|RomanceComedy|RomanceCome...,30056.0,moldyoldmoldyoldmoldyoldmoldyoldmoldyoldmoldyo...,1.189162e+11
4,3078,33.0,12580104096,804195.0,219499.0,Waiting to Exhale (1995)Waiting to Exhale (199...,Comedy|Drama|RomanceComedy|Drama|RomanceComedy...,0.0,0,0.000000e+00
5,58716,602.0,194562210376,5539009.0,581238.0,Father of the Bride Part II (1995)Father of th...,ComedyComedyComedyComedyComedyComedyComedyCome...,46452.0,pregnancyremakepregnancyremakepregnancyremakep...,1.114626e+11
...,...,...,...,...,...,...,...,...,...,...
193581,368,8.0,3074218164,5476944.0,432131.0,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,0.0,0,0.000000e+00
193583,368,7.0,3074219090,5914996.0,445030.0,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,0.0,0,0.000000e+00
193585,368,7.0,3074219610,6397426.0,479308.0,Flint (2017),Drama,0.0,0,0.000000e+00


## Menggabungkan data dengan fitur title film

In [17]:
all_movie_rate = ratings[['userId', 'movieId', 'rating']]
all_movie_rate

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [18]:
all_movie_name = pd.merge(all_movie_rate, movies[['movieId','title', 'genres']], on='movieId', how='left')

all_movie_name

,userId,movieId,rating,title,genres
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...
100831,610,166534,4.0,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,Get Out (2017),Horror
100834,610,168252,5.0,Logan (2017),Action|Sci-Fi


## Menggabungkan data dengan fitur tag film

In [19]:
all_movie = pd.merge(all_movie_name, tags[['movieId','tag']], on='movieId', how='left')
all_movie

,userId,movieId,rating,title,genres,tag
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
1,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
2,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,fun
3,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance,moldy
4,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance,old
...,...,...,...,...,...,...
285757,610,168252,5.0,Logan (2017),Action|Sci-Fi,emotional
285758,610,168252,5.0,Logan (2017),Action|Sci-Fi,gritty
285759,610,168252,5.0,Logan (2017),Action|Sci-Fi,heartbreaking
285760,610,168252,5.0,Logan (2017),Action|Sci-Fi,predictible plot


# Data Preparation

## Mengatasi missing value

In [20]:
all_movie.isnull().sum()

,0
userId,0
movieId,0
rating,0
title,0
genres,0
tag,52549


Dari informasi diatas menunjukkan jumlah missing value (nilai kosong) pada kolom tag mencapai 52.549 nilai kosong, sedangkan kolom lain (userId, movieId, rating, title, dan genres) tidak memiliki missing value. Data yang hilang atau kosonga akan dihapus.

In [21]:
all_movie_clean = all_movie.dropna()
all_movie_clean

,userId,movieId,rating,title,genres,tag
0,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
1,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
2,1,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,fun
3,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance,moldy
4,1,3,4.0,Grumpier Old Men (1995),Comedy|Romance,old
...,...,...,...,...,...,...
285756,610,168252,5.0,Logan (2017),Action|Sci-Fi,dark
285757,610,168252,5.0,Logan (2017),Action|Sci-Fi,emotional
285758,610,168252,5.0,Logan (2017),Action|Sci-Fi,gritty
285759,610,168252,5.0,Logan (2017),Action|Sci-Fi,heartbreaking


In [22]:
all_movie_clean.isnull().sum()

,0
userId,0
movieId,0
rating,0
title,0
genres,0
tag,0


Setelah menghapus data yang hilang atau kosong kini data sudah bersih yang berjumlah 230k baris data.

## Cek ulang data film

In [23]:
fix_movie = all_movie_clean.sort_values('movieId', ascending=True)
fix_movie.head()

,userId,movieId,rating,title,genres,tag
20958,46,1,5.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
20959,46,1,5.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,fun
35188,73,1,4.5,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
35189,73,1,4.5,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
35190,73,1,4.5,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,fun


In [24]:
print('Jumlah film: ',len(fix_movie.movieId.unique()))
print('Jumlah jenis rating: ',len(fix_movie.rating.unique()))
print('Jumlah genre film: ', len(fix_movie.genres.unique()))
print('Jumlah tag/ label film: ', len(fix_movie.tag.unique()))

Jumlah film:  1554
Jumlah jenis rating:  10
Jumlah genre film:  370
Jumlah tag/ label film:  1584


## Tahapan persiapan

In [25]:
preparation = fix_movie
preparation.sort_values('movieId').head()

,userId,movieId,rating,title,genres,tag
237587,517,1,4.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
186264,411,1,5.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
186265,411,1,5.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
186266,411,1,5.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,fun
61091,134,1,3.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar


### Membuang data duplikat

In [26]:
preparation = preparation.drop_duplicates('movieId')
preparation

,userId,movieId,rating,title,genres,tag
20958,46,1,5.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
221129,476,2,4.0,Jumanji (1995),Adventure|Children|Fantasy,magic board game
99728,226,3,3.5,Grumpier Old Men (1995),Comedy|Romance,old
53201,117,5,3.0,Father of the Bride Part II (1995),Comedy,remake
75977,177,7,1.0,Sabrina (1995),Comedy|Romance,remake
...,...,...,...,...,...,...
27633,62,183611,4.0,Game Night (2018),Action|Comedy|Crime|Horror,funny
27637,62,184471,3.5,Tomb Raider (2018),Action|Adventure|Fantasy,video game adaptation
151991,331,187593,4.0,Deadpool 2 (2018),Action|Comedy|Sci-Fi,Josh Brolin
174270,380,187595,4.0,Solo: A Star Wars Story (2018),Action|Adventure|Children|Sci-Fi,star wars


### Konversi data series jadi list

In [27]:
movie_id = preparation['movieId'].tolist()

movie_title = preparation['title'].tolist()

movie_tag = preparation['tag'].tolist()

movie_genre = preparation['genres'].tolist()

print(len(movie_id))
print(len(movie_title))
print(len(movie_tag))
print(len(movie_genre))

1554
1554
1554
1554


Bisa dilihat sekarang semua data memiliki jumlah yang sama.

### Membuat dictionary

In [28]:
movie_new = pd.DataFrame({
        'id': movie_id,
        'title': movie_title,
        'genre': movie_genre,
        'tag': movie_tag
    })
movie_new

,id,title,genre,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar
1,2,Jumanji (1995),Adventure|Children|Fantasy,magic board game
2,3,Grumpier Old Men (1995),Comedy|Romance,old
3,5,Father of the Bride Part II (1995),Comedy,remake
4,7,Sabrina (1995),Comedy|Romance,remake
...,...,...,...,...
1549,183611,Game Night (2018),Action|Comedy|Crime|Horror,funny
1550,184471,Tomb Raider (2018),Action|Adventure|Fantasy,video game adaptation
1551,187593,Deadpool 2 (2018),Action|Comedy|Sci-Fi,Josh Brolin
1552,187595,Solo: A Star Wars Story (2018),Action|Adventure|Children|Sci-Fi,star wars


# Model Development

## Content Based Filtering

### Mendefinisikan data yang sudah bersih kedalam variabel

In [29]:
data = movie_new

### TF-IDF Vectorizer

In [30]:
tf = TfidfVectorizer()
tf.fit(data['genre'])
tf.get_feature_names_out()

array(['action', 'adventure', 'animation', 'children', 'comedy', 'crime',
       'documentary', 'drama', 'fantasy', 'fi', 'film', 'genres',
       'horror', 'imax', 'listed', 'musical', 'mystery', 'no', 'noir',
       'romance', 'sci', 'thriller', 'war', 'western'], dtype=object)

In [31]:
tfidf_matrix = tf.fit_transform(data['genre'])
tfidf_matrix.shape

(1554, 24)

Setelah disimpan dalam bentuk matrix data yang diperoleh sebanyak 1554 baris dengan 24 feature

In [32]:
tfidf_matrix.todense()

matrix([[0.        , 0.3983147 , 0.52247057, ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.49497089, 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        ...,
        [0.49413097, 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.39639614, 0.40792706, 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.41107716, 0.        , 0.55489644, ..., 0.        , 0.        ,
         0.        ]])

In [33]:
pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=data.title
).sample(10, axis=1).sample(5, axis=0)

,thriller,adventure,animation,mystery,children,war,musical,no,western,sci
title,,,,,,,,,,
Double Indemnity (1944),0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
"Rainmaker, The (1997)",0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
It Takes Two (1995),0.0,0.0,0.0,0.0,0.875445,0.0,0.0,0.0,0.0,0.0
"Woman Under the Influence, A (1974)",0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
Splash (1984),0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


Nilai 0.0 artinya genre tersebut tidak relevan atau tidak disebutkan dalam deskripsi genre film itu.

Nilai > 0 menunjukkan tingkat kepentingan atau kemunculan genre tersebut dalam data genre film. Semakin tinggi nilainya, semakin penting genre tersebut untuk film itu.

Macbeth (1971) memiliki nilai 1.0 pada drama, artinya genre "drama" sangat dominan atau satu-satunya genre yang muncul.

Arsenic and Old Lace (1944) memiliki nilai TF-IDF 0.549942 untuk thriller, artinya genre "thriller" relevan untuk film itu, meski mungkin tidak satu-satunya.

## Cosine Similarity

Cosine similarity mengukur kesamaan antara dua vektor dan menentukan apakah kedua vektor tersebut menunjuk ke arah yang sama. Ia menghitung sudut cosinus antara dua vektor. Semakin kecil sudut cosinus, semakin besar nilai cosine similarity.

In [34]:
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

array([[1.        , 0.80472348, 0.1808617 , ..., 0.10142228, 0.42942348,
        0.37429223],
       [0.80472348, 1.        , 0.        , ..., 0.        , 0.53362862,
        0.        ],
       [0.1808617 , 0.        , 1.        , ..., 0.23089549, 0.        ,
        0.19208644],
       ...,
       [0.10142228, 0.        , 0.23089549, ..., 1.        , 0.6983388 ,
        0.83191943],
       [0.42942348, 0.53362862, 0.        , ..., 0.6983388 , 1.        ,
        0.58096161],
       [0.37429223, 0.        , 0.19208644, ..., 0.83191943, 0.58096161,
        1.        ]])

In [35]:
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['title'], columns=data['title'])
print('Shape:', cosine_sim_df.shape)
cosine_sim_df.sample(5, axis=1).sample(5, axis=0)

Shape: (1554, 1554)


title,Unbreakable (2000),Blade Runner 2049 (2017),Gross Anatomy (a.k.a. A Cut Above) (1989),"Sword in the Stone, The (1963)",Anchorman 2: The Legend Continues (2013)
title,,,,,
Primer (2004),1.0,0.946397,0.193915,0.0,0.000000
Hoop Dreams (1994),0.0,0.000000,0.000000,0.0,0.000000
Me Myself I (2000),0.0,0.000000,0.513174,0.0,0.641674
My Flesh and Blood (2003),0.0,0.000000,0.000000,0.0,0.000000
Our Hospitality (1923),0.0,0.000000,0.799743,0.0,1.000000


Shape (1554, 1554) merupakan ukuran matriks similarity dari data yang kita miliki. Berdasarkan data yang ada, matriks di atas sebenarnya berukuran 1554 film x 1554 film (masing-masing dalam sumbu X dan Y). Artinya, kita mengidentifikasi tingkat kesamaan pada 1554 nama film. Tapi tentu kita tidak bisa menampilkan semuanya. Oleh karena itu, kita hanya memilih 5 film pada baris vertikal dan 5 film pada sumbu horizontal seperti pada contoh di atas.

Angka 1.0 mengindikasikan bahwa judul film pada kolom X (horizontal) memiliki kesamaan dengan judul film pada baris Y (vertikal). Sebagai contoh, judul film Michael Collins (1996) teridentifikasi sama (similar) dengan judul film Beyond Silence (Jenseits der Stille) (1996).

## Mendapatkan Rekomendasi

In [36]:
def movie_recommendations(title, similarity_data=cosine_sim_df, items=data[['title', 'genre']], k=10):
    index = similarity_data.loc[:, title].to_numpy().argpartition(range(-1, -k, -1))
    closest = similarity_data.columns[index[-1:-(k+2):-1]]
    closest = closest.drop(title, errors='ignore')
    return pd.DataFrame(closest).merge(items).head(k)


In [37]:
data[data.title.eq('My Life (1993)')]

,id,title,genre,tag
719,3502,My Life (1993),Drama,death


Dari informasi diatas dapat kita lihat bahwa film My Life termasuk dalam film bergenre drama

In [38]:
movie_recommendations('My Life (1993)')

,title,genre
0,Friendly Persuasion (1956),Drama
1,Luther (2003),Drama
2,Ikiru (1952),Drama
3,"Gospel According to St. Matthew, The (Vangelo ...",Drama
4,Lost Horizon (1937),Drama
5,Cat on a Hot Tin Roof (1958),Drama
6,Seabiscuit (2003),Drama
7,"Secret Lives of Dentists, The (2002)",Drama
8,"Magdalene Sisters, The (2002)",Drama
9,Anastasia (1956),Drama


Untuk rekomendasinya sendiri sudah tepat dimana sistem merekomendasikan top 5 judul film yang memiliki genre sama dengan film My Life yaitu :
* Friendly Persuasion (1956)
* Luther (2003)
* Ikiru (1952)
* Gospel According to St. Matthew, The (Vangelo secondo Matteo) (1964)
* Lost Horizon (1937)
* Cat on a Hot Tin Roof (1958)
* Seabiscuit (2003)
* Secret Lives of Dentists, The (2002)
* Magdalene Sisters, The (2002)
* Anastasia (1956)

# Evaluation

## Precision matrix

![Alt Text](https://assets.cdn.dicoding.com/original/academy/dos:819311f78d87da1e0fd8660171fa58e620211012160253.png)

* **Jumlah rekomendasi yang relevan:** Berdasarkan asumsi kita, semua 10 film bergenre "Drama" dianggap relevan. Jadi, jumlah rekomendasi yang relevan adalah 10.
* **Jumlah item yang direkomendasikan:** Ada 10 film dalam daftar rekomendasi.

Maka, persisinya adalah:

$\qquad P = \frac{10}{10} = 1$

**Kesimpulan:**

Berdasarkan asumsi bahwa semua film bergenre "Drama" relevan, matriks persisi untuk rekomendasi film setelah menonton "My Life (1993)" adalah **1**. Ini berarti bahwa 100% film yang direkomendasikan dianggap relevan berdasarkan genre.